### Assignment: Evolutionary Algorithm for Timetable Scheduling

#### Objective
Your task is to develop an **Evolutionary Algorithm (EA)** that optimizes a timetable for a set of courses based on predefined constraints. The algorithm should generate a valid schedule that meets the specified requirements while optimizing for minimal conflicts and balanced distribution of classes.

#### Problem Description
You are given a file (`classes.json`) containing a set of courses, their class types, scheduled days, and start times. Your goal is to create a feasible timetable that schedules all required classes while adhering to the given constraints.

#### Constraints
1. **Total Classes**: The timetable must contain **exactly 11 classes**.
2. **Course Distribution**:
   - **Tópicos de Física Moderna (TFM)**: 3 classes (**2 T1** and **1 TP**)
   - **Princípios de Programação Procedimental (PPP)**: 2 classes (**2 TP**)
   - **Comunicação Técnica (CT)**: 2 classes (**1 T1** and **1 PL**)
   - **Estatística**: 2 classes (**2 TP**)
   - **Análise Matemática II (AMII)**: 2 classes (**2 TP**)
3. **No Overlaps**: Two classes cannot be scheduled at the same time slot.
4. **Valid Time Slots**: Each class can only be scheduled in one of the available time slots provided in `classes.json`.

#### Input Format
The input file `classes.json` contains an array of objects, where each object has the following attributes:
- **Course**: The name of the course.
- **Class**: The type and section of the class (e.g., T1, TP1, PL1).
- **Day**: The scheduled day of the week.
- **Start Time**: The starting time of the class.

Example JSON entry:
```json
{
    "Course": "Análise Matemática II",
    "Class": "TP1",
    "Day": "Tuesday",
    "Start Time": "09:00"
}
```

#### Evolutionary Algorithm Requirements
Your **Evolutionary Algorithm** should follow these key principles:
1. **Representation**: Design an appropriate encoding for the timetable.
2. **Fitness Function**: Evaluate solutions based on:
   - Validity (whether all constraints are met)
   - Minimized conflicts
   - Balanced distribution of classes across time slots
3. **Parent Selection**: Implement a method for selecting promising solutions. Start by implementing the tournament selection.
4. **Crossover**: Define a mechanism to combine two parent solutions to create new timetables.
5. **Mutation**: Implement a mutation strategy to introduce diversity.
6. **Termination Condition**: Decide when the algorithm should stop (e.g., after a fixed number of generations or when there is no significant improvement).

After programming all of this you should implement the elitism mechanism.


Good luck, and happy coding!


In [11]:
# import libraries and load data from json
import random
import json
from datetime import datetime
import copy

NUMBER_OF_CLASSES_PER_WEEK = 11

# Load class data from JSON file
with open('classes.json', 'r', encoding='utf-8') as f:
    class_data = json.load(f)

days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
times = ["09:00", "11:00", "14:00", "16:00", "18:00"]
courses_classe = {}
for i in class_data:
    courses_classe.setdefault(i['Course'], set()).add(i['Class'])

total_courses = len(courses_classe)
max_classes = max(len(classes) for classes in courses_classe.values())
total_days = len(days)
total_time_slots = len(times)

In [12]:
# check the contents:

## to check the given options for timetable selection
for c in class_data:
    print(c)

print("....")
## to access courses and classes
for cor in courses_classe:
    print(cor, " -> ", courses_classe[cor])
print(len(class_data))

{'Course': 'Análise Matemática II', 'Class': 'TP1', 'Day': 'Tuesday', 'Start Time': '09:00'}
{'Course': 'Análise Matemática II', 'Class': 'TP2', 'Day': 'Tuesday', 'Start Time': '09:00'}
{'Course': 'Análise Matemática II', 'Class': 'TP3', 'Day': 'Tuesday', 'Start Time': '09:00'}
{'Course': 'Estatística', 'Class': 'TP3', 'Day': 'Tuesday', 'Start Time': '09:00'}
{'Course': 'Comunicação Técnica', 'Class': 'T1', 'Day': 'Wednesday', 'Start Time': '09:00'}
{'Course': 'Análise Matemática II', 'Class': 'TP1', 'Day': 'Thursday', 'Start Time': '09:00'}
{'Course': 'Análise Matemática II', 'Class': 'TP2', 'Day': 'Thursday', 'Start Time': '09:00'}
{'Course': 'Análise Matemática II', 'Class': 'TP3', 'Day': 'Thursday', 'Start Time': '09:00'}
{'Course': 'Estatística', 'Class': 'TP3', 'Day': 'Thursday', 'Start Time': '09:00'}
{'Course': 'Tópicos de Física Moderna', 'Class': 'T1', 'Day': 'Monday', 'Start Time': '09:00'}
{'Course': 'Tópicos de Física Moderna', 'Class': 'T1', 'Day': 'Friday', 'Start Time':

# Initial Solution
Generates a random a chromosome that will represent a timetable

In [13]:
def generate_chromosome():
  return random.sample(range(len(class_data)), NUMBER_OF_CLASSES_PER_WEEK)

In [14]:
def chromosome_to_timetable(chromosome): # generate the timetable object from the representation
    ''' from the genes you should generate a list with elements that
    follow a structure similar to classes.json as follows:
        {
            "Course": course,
            "Class": classe,
            "Day": day,
            "Start Time": time
        }'''
    return [class_data[i] for i in chromosome]

In [15]:
# helper function to export timetables
def save_timetable(timetable, filename='a_timetable.json',fitness_value="not calculated"):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(sorted(timetable, key=lambda x: x['Course']) + [{'fitness' : fitness_value}], f, indent=4, ensure_ascii=False)


# Fitness function
Evaluates the quality of the timetable encoded in each solution. Solutions that do not take into account the restrictions of the problem should be penalized. Lower values are better.

In [16]:
def fitness(chromosome):
    '''

1. **Total Classes**: The timetable must contain **exactly 11 classes**.
2. **Course Distribution**:
   - **Tópicos de Física Moderna (TFM)**: 3 classes (**2 T1** and **1 TP**)
   - **Princípios de Programação Procedimental (PPP)**: 2 classes (**2 TP**)
   - **Comunicação Técnica (CT)**: 2 classes (**1 T1** and **1 PL**)
   - **Estatística**: 2 classes (**2 TP**)
   - **Análise Matemática II (AMII)**: 2 classes (**2 TP**)
3. **No Overlaps**: Two classes cannot be scheduled at the same time slot.
4. **Valid Time Slots**: Each class can only be scheduled in one of the available time slots provided in `classes.json`.

    '''

    schedule = chromosome_to_timetable(chromosome)

    penalty = 0

    seen_times = set()

    for c in schedule:
      time_slot = (c['Day'], c['Start Time'])

      if time_slot in seen_times:
          penalty += 100
      else:
          seen_times.add(time_slot)

    counts = {
        "TFM_T1": 0, "TFM_TP": 0,
        "PPP_TP": 0,
        "CT_T1": 0, "CT_PL": 0,
        "EST_TP": 0,
        "AMII_TP": 0
    }

    for c in schedule:
        course = c["Course"]
        c_type = c["Class"]

        if course == "Tópicos de Física Moderna":
            if "T1" in c_type: counts["TFM_T1"] += 1
            elif "TP" in c_type: counts["TFM_TP"] += 1

        elif course == "Princípios de Programação Procedimental":
            if "TP" in c_type: counts["PPP_TP"] += 1

        elif course == "Comunicação Técnica":
            if "T1" in c_type: counts["CT_T1"] += 1
            elif "PL" in c_type: counts["CT_PL"] += 1

        elif course == "Estatística":
            if "TP" in c_type: counts["EST_TP"] += 1

        elif course == "Análise Matemática II":
            if "TP" in c_type: counts["AMII_TP"] += 1

    penalty += abs(counts["TFM_T1"] - 2) * 10
    penalty += abs(counts["TFM_TP"] - 1) * 10

    penalty += abs(counts["PPP_TP"] - 2) * 10

    penalty += abs(counts["CT_T1"] - 1) * 10
    penalty += abs(counts["CT_PL"] - 1) * 10

    penalty += abs(counts["EST_TP"] - 2) * 10

    penalty += abs(counts["AMII_TP"] - 2) * 10

    days_used = set()

    # for c in schedule:
    #  days_used.add(c['Day'])

    # penalty += len(days_used) * 5

    # ----------------------------
    # penalty for gaps in schedule
    # ----------------------------

    daily_hours = {"Monday": [], "Tuesday": [], "Wednesday": [], "Thursday": [], "Friday": []}

    for c in schedule:
      hour = int(c['Start Time'][:2])
      daily_hours[c['Day']].append(hour)

    for day, hours in daily_hours.items():
      if len(hours) > 1:
        hours.sort()

        for i in range(len(hours) - 1):
          jump = hours[i + 1] - hours[i]

        if jump > 3:
          penalty += (jump - 3) * 5

    return penalty





  # Crossover
You should program a crossover operator.

In [17]:
def crossover(parent1, parent2):
    cut = random.randint(1, NUMBER_OF_CLASSES_PER_WEEK - 1)
    child = parent1[:cut]

    for class_id in parent2:
      if class_id not in child:
        child.append(class_id)

      if len(child) == NUMBER_OF_CLASSES_PER_WEEK:
        break

    return child


# Mutation operator
You should program a crossover operator.

In [18]:
def mutate(individual, mutation_rate):
    if random.random() < mutation_rate:

        drop_index = random.randint(0, NUMBER_OF_CLASSES_PER_WEEK - 1)

        unused_classes = []
        for i in range(len(class_data)):
            if i not in individual:
                unused_classes.append(i)

        new_class = random.choice(unused_classes)

        individual[drop_index] = new_class

    return individual

# Parent Selection
You should program the tournament selection mechanism

In [19]:
def tournament_selection(population, k=5):
    tourney_1 = random.sample(population, k)
    p1 = sorted(tourney_1, key=fitness)[0]

    tourney_2 = random.sample(population, k)
    p2 = sorted(tourney_2, key=fitness)[0]

    return p1, p2



In [20]:

def genetic_algorithm(pop_size=100, generations=2500, mutation_rate=0.05):
    population = [generate_chromosome() for _ in range(pop_size)]
    for gen in range(generations):
        population = sorted(population, key=fitness)
        if fitness(population[0]) == 0:
            break
        new_population = []

        new_population.append(population[0])

        while len(new_population) < pop_size:
            p1, p2 = tournament_selection(population)
            child = crossover(p1, p2) # cuidado com as alterações usem np.copy ou equivalente
            child = mutate(child, mutation_rate)
            new_population.append(child)
        population = new_population
        print(f"Generation {gen + 1}, Best Fitness: {fitness(population[0])}")
    return population[0]

for r in range(10):
    random.seed(2025+r)
    best_timetable = genetic_algorithm()

final_schedule = chromosome_to_timetable(best_timetable)

day_order = {"Monday": 1, "Tuesday": 2, "Wednesday": 3, "Thursday": 4, "Friday": 5}

sorted_schedule = sorted(final_schedule, key=lambda c: (day_order[c["Day"]], c["Start Time"]))

print(" PERFECT TIMETABLE FOUND \n")

for c in sorted_schedule:
    print(f"{c['Day']} at {c['Start Time']} - {c['Course']} ({c['Class']})")

Generation 1, Best Fitness: 180
Generation 2, Best Fitness: 110
Generation 3, Best Fitness: 280
Generation 4, Best Fitness: 75
Generation 5, Best Fitness: 100
Generation 6, Best Fitness: 40
Generation 7, Best Fitness: 90
Generation 8, Best Fitness: 80
Generation 9, Best Fitness: 40
Generation 10, Best Fitness: 80
Generation 11, Best Fitness: 140
Generation 12, Best Fitness: 30
Generation 13, Best Fitness: 140
Generation 14, Best Fitness: 40
Generation 15, Best Fitness: 50
Generation 16, Best Fitness: 45
Generation 17, Best Fitness: 20
Generation 18, Best Fitness: 140
Generation 19, Best Fitness: 40
Generation 20, Best Fitness: 10
Generation 21, Best Fitness: 120
Generation 22, Best Fitness: 50
Generation 23, Best Fitness: 40
Generation 24, Best Fitness: 140
Generation 25, Best Fitness: 120
Generation 26, Best Fitness: 50
Generation 27, Best Fitness: 20
Generation 28, Best Fitness: 50
Generation 1, Best Fitness: 260
Generation 2, Best Fitness: 60
Generation 3, Best Fitness: 185
Generati